# 03 - Training/validation split of ESM25

ESM25 is used for training and validation only; the test set will be made of later ESM records (events after the last
ESM25 event and not present in ESM25).

The split is grouped by event and stratified by label: all records of an event fall in the same set, so the validation
set contains only unseen events, as the test set will. Stations are largely shared between the sets, as they will be with
the test set: the results must be read in this light (or evaluated separately on stations not seen in training).

Parameters in the `split` section of `configs/default.yaml`. The split is saved once and reused by every experiment.

## 0. Setup

In [ ]:
from pathlib import Path

import pandas as pd

from wqu.config import load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.loading import ORIGIN_TIME_COLUMN, load_esm25_metadata
from wqu.data.splitting import (
    SPLIT_COLUMN,
    assign_split,
    save_split,
    shared_events,
    split_counts,
    station_overlap,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
columns = config.columns
split = config.split
label_names = {subset.label: subset.name for subset in config.data.subsets}
split.tables_dir.mkdir(parents=True, exist_ok=True)

_, metadata = load_esm25_metadata(config)
print(f"Records: {len(metadata):,}; events: {metadata[columns.event_id].nunique():,}; "
      f"stations: {metadata[columns.station_id].nunique():,}")
print(f"Validation fraction {split.validation_fraction:g} -> {split.n_folds} folds, the first is the validation set; "
      f"seed {split.random_seed}")

## 1. Assign and save the split

In [ ]:
split_table = assign_split(metadata, columns, split)
split_path = save_split(split_table, split.output_file)
print(f"Split saved in {split_path}")
counts = split_counts(split_table, columns)
counts.to_csv(split.tables_dir / "split_counts.csv")
counts.round(3)

## 2. Checks

- no event in both sets;
- how much the validation stations are known from training;
- class balance and main physical variables in the two sets (they should be similar).

In [ ]:
n_shared_events = shared_events(split_table, columns)
if n_shared_events:
    raise RuntimeError(f"{n_shared_events} events are in both sets.")
print("Events shared by the two sets: 0")
pd.Series(station_overlap(split_table, columns)).round(3)

In [ ]:
per_label = split_table.groupby([SPLIT_COLUMN, columns.label]).size().unstack(columns.label).rename(columns=label_names)
per_label

In [ ]:
COMPARED_COLUMNS = ["preferred_magnitude", "path_hyp_dist_km", "origin_year"]
with_split = metadata.join(split_table[SPLIT_COLUMN])
comparison = numeric_summary_by_label(with_split, COMPARED_COLUMNS, SPLIT_COLUMN)
comparison.to_csv(split.tables_dir / "split_variables.csv")
comparison.round(2)

In [ ]:
bad_rate_by_year = (
    with_split.assign(year=with_split[ORIGIN_TIME_COLUMN].dt.year)
    .groupby(["year", SPLIT_COLUMN])[columns.label]
    .mean()
    .unstack(SPLIT_COLUMN)
)
bad_rate_by_year.tail(10).round(3)